In [ ]:
import cv2
import numpy as np
import os
import random

# Augmentation
def random_params():
    angle = random.uniform(-10, 10)       # degrees
    scale = random.uniform(0.94, 1.06)
    tx = random.uniform(-0.05, 0.05)      # fraction of width
    ty = random.uniform(-0.05, 0.05)      # fraction of height
    return angle, scale, tx, ty

def augment_frame(frame, angle, scale, tx, ty):
    h, w = frame.shape[:2]
    cx, cy = w/2, h/2
    M = cv2.getRotationMatrix2D((cx, cy), angle, scale)
    M[0,2] += tx * w
    M[1,2] += ty * h
    transformed = cv2.warpAffine(frame, M, (w,h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    return transformed, M

def transform_keypoints(kps, M):
    kps_h = np.concatenate([kps, np.ones((len(kps),1))], axis=1)
    transformed = (M @ kps_h.T).T
    return transformed[:, :2]

# Process
for file in os.listdir(input_folder):
    if not file.endswith(".mp4"):
        continue
    cap = cv2.VideoCapture(os.path.join(input_folder, file))
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()

    angle, scale, tx, ty = random_params()
    aug_frames = []
    for f in frames:
        aug_f, M = augment_frame(f, angle, scale, tx, ty)
        aug_frames.append(aug_f)


    # save
    h, w = aug_frames[0].shape[:2]
    out_path = os.path.join(output_folder, f"aug_{file}")
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(out_path, fourcc, 30, (w, h))
    for f in aug_frames:
        out.write(f)
    out.release()
    print(f"Saved {out_path}")
